# AMD Hackathon — Vulnerability-Aware Java LLM: GRPO Training

Fine-tunes **Qwen3-8B** on our Java security dataset using **GRPO** (Group Relative Policy Optimization).

**Pipeline:**
1. Load model + apply custom reasoning chat template
2. **Pre-SFT warmup** on `output/train_chat_template.jsonl` — teaches the `<start_analysis>...<FIXED_CODE>` format
3. **GRPO training** with security-aware reward functions
4. Inference test + save

**Dataset:** `output/train_chat_template.jsonl` — 143 Java CVE fix pairs from CWE-Bench-Java  
**Model output:** `output/qwen3-java-vuln-grpo/lora_adapter`

### Installation

In [ ]:
import sys, subprocess
from pathlib import Path

VENV_DIR = Path('.venv-grpo')
KERNEL_NAME = 'grpo-venv'
DISPLAY_NAME = 'Python (grpo-venv)'

# Install uv + ipykernel in the current bootstrap environment
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'uv', 'ipykernel'])

# Create venv if needed
if not VENV_DIR.exists():
    subprocess.check_call([sys.executable, '-m', 'uv', 'venv', str(VENV_DIR)])

venv_python = VENV_DIR / 'bin' / 'python'
subprocess.check_call([str(venv_python), '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel', 'ipykernel'])
subprocess.check_call([
    str(venv_python), '-m', 'ipykernel', 'install', '--user',
    '--name', KERNEL_NAME, '--display-name', DISPLAY_NAME, '--force'
])

print('Venv ready:', VENV_DIR.resolve())
print('Kernel registered:', DISPLAY_NAME)
print('Now switch the notebook kernel to "Python (grpo-venv)" and continue.')

In [ ]:
import sys
if '.venv-grpo' not in sys.executable:
    raise RuntimeError(
        'This notebook is not running on .venv-grpo. Switch kernel to "Python (grpo-venv)" and re-run.'
    )
print('Active interpreter:', sys.executable)

### Notebook Environment Bootstrap (run once)

This notebook now uses a dedicated virtual environment to avoid cloud package conflicts.

1. Run the next cell to create `.venv-grpo` and register a Jupyter kernel.
2. Switch kernel to **Python (grpo-venv)**.
3. Re-run from the installation cell onward.

In [ ]:
%%capture
import os

# Base tool for fast installs
!pip install --upgrade -qqq uv

# Your requested install flow
if "COLAB_" not in "".join(os.environ.keys()):
    # Non-Colab path: keep it minimal and ROCm-safe
    !pip install unsloth
else:
    try:
        import numpy, PIL
        _numpy = f'numpy=={numpy.__version__}'
        _pil = f'pillow=={PIL.__version__}'
    except Exception:
        _numpy = 'numpy'
        _pil = 'pillow'
    try:
        import subprocess
        is_t4 = 'Tesla T4' in str(subprocess.check_output(['nvidia-smi']))
    except Exception:
        is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade 'torchao>=0.16.0'

# Keep your ROCm vLLM if already present, only align HF/TRL stack
!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps trl==0.22.2

In [ ]:
import importlib, sys

def _ver(pkg: str):
    try:
        return importlib.import_module(pkg).__version__
    except Exception as e:
        return f'NOT_INSTALLED ({e.__class__.__name__})'

print('Python      :', sys.version.split()[0])
print('transformers:', _ver('transformers'))
print('trl         :', _ver('trl'))
print('unsloth     :', _ver('unsloth'))
print('vllm        :', _ver('vllm'))

# Compatibility note for this notebook stack:
print('\nExpected for this notebook setup:')
print('- transformers 4.56.2')
print('- trl 0.22.2')
print('- vLLM ROCm build kept as preinstalled (example: 0.11.0rc2...rocm700)')

### Config — edit before running

In [ ]:
import gc, json, os, re, sys
from pathlib import Path
import torch

MODEL_NAME          = "unsloth/Qwen3-8B-unsloth-bnb-4bit"  # swap to Qwen3-4B for smaller GPU
DATASET_PATH        = "output/train_chat_template.jsonl"
OUTPUT_DIR          = "output/qwen3-java-vuln-grpo"

MAX_SEQ_LENGTH      = 4096
LORA_RANK           = 32

# Pre-SFT warmup
SFT_EPOCHS          = 3
SFT_LR              = 2e-4

# GRPO
GRPO_MAX_STEPS      = 200   # increase to 500 for a serious run
GRPO_LR             = 5e-6
GRPO_NUM_GEN        = 4     # reduce to 2 if OOM
GRPO_MAX_NEW_TOKENS = 1024
GRPO_TEMPERATURE    = 0.9

# ── ROCm / AMD GPU environment ────────────────────────────────────────────────
os.environ.setdefault("PYTORCH_HIP_ALLOC_CONF", "expandable_segments:True")
# Disable SDMA to avoid transfer stalls on some AMD GPU configs
os.environ.setdefault("HSA_ENABLE_SDMA", "0")
# ─────────────────────────────────────────────────────────────────────────────

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Config loaded. Dataset: {DATASET_PATH}  Model: {MODEL_NAME}")
print(f"ROCm available: {torch.cuda.is_available()}  "
      f"(torch sees AMD GPU via HIP/ROCm as 'cuda' backend)")

In [ ]:
# ── AMD GPU Monitoring & Throughput Tracking ──────────────────────────────────
# PyTorch on ROCm exposes AMD GPUs through the torch.cuda namespace (HIP backend).
# All torch.cuda.* memory calls work identically on ROCm.
import time as _time
import torch

# Some cloud images expose a partially broken top-level transformers import path.
# Fallback to the module-local import to avoid TrainerCallback import errors.
try:
    from transformers import TrainerCallback
except Exception:
    from transformers.trainer_callback import TrainerCallback

class GPUMonitor:
    """Lightweight AMD GPU memory + utilization tracker via PyTorch ROCm/HIP backend."""

    @staticmethod
    def stats() -> dict:
        if not torch.cuda.is_available():
            return {}
        out = {}
        for i in range(torch.cuda.device_count()):
            props  = torch.cuda.get_device_properties(i)
            used   = torch.cuda.memory_allocated(i) / 1024**3
            reserv = torch.cuda.memory_reserved(i)  / 1024**3
            total  = props.total_memory / 1024**3
            out[i] = dict(
                name      = props.name,
                used_gb   = round(used, 2),
                reserv_gb = round(reserv, 2),
                total_gb  = round(total, 2),
                free_gb   = round(total - reserv, 2),
                util_pct  = round(used / total * 100, 1),
            )
        return out

    @staticmethod
    def print_stats(label: str = ""):
        tag = f"[{label}] " if label else ""
        for gpu_id, s in GPUMonitor.stats().items():
            bar_n = int(s["util_pct"] / 5)
            bar   = "█" * bar_n + "░" * (20 - bar_n)
            print(f"  {tag}GPU {gpu_id} ({s['name']}):  "
                  f"{s['used_gb']:.1f}/{s['total_gb']:.1f} GB  "
                  f"|{bar}| {s['util_pct']:.0f}%  ({s['free_gb']:.1f} GB free)")

    @staticmethod
    def reset_peak(device: int = 0):
        torch.cuda.reset_peak_memory_stats(device)

    @staticmethod
    def peak_gb(device: int = 0) -> float:
        return torch.cuda.max_memory_allocated(device) / 1024**3


class ThroughputCallback(TrainerCallback):
    """Logs steps/sec, samples/sec, and peak VRAM every N training steps."""

    def __init__(self, log_every: int = 10):
        self._t0       = None
        self._step0    = 0
        self.log_every = log_every
        self.history: list[dict] = []

    def on_train_begin(self, args, state, control, **kwargs):
        self._t0    = _time.time()
        self._step0 = 0
        GPUMonitor.reset_peak()
        GPUMonitor.print_stats("train_begin")

    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        if step % self.log_every != 0 or self._t0 is None:
            return
        elapsed = _time.time() - self._t0
        steps   = step - self._step0
        if elapsed <= 0 or steps <= 0:
            return
        sps  = steps / elapsed
        smps = sps * args.per_device_train_batch_size * args.gradient_accumulation_steps
        peak = GPUMonitor.peak_gb()
        rec  = dict(step=step, steps_per_sec=round(sps, 3),
                    samples_per_sec=round(smps, 3), peak_vram_gb=round(peak, 2))
        self.history.append(rec)
        print(f"  📊 step {step:>4d} | {sps:.2f} steps/s | "
              f"{smps:.2f} samp/s | peak VRAM {peak:.1f} GB")
        self._t0    = _time.time()
        self._step0 = step

    def on_train_end(self, args, state, control, **kwargs):
        print(f"\n  Peak VRAM during training: {GPUMonitor.peak_gb():.2f} GB")
        GPUMonitor.print_stats("train_end")


throughput_cb = ThroughputCallback(log_every=10)

# Show AMD GPU state at startup
GPUMonitor.print_stats("startup")
print(f"  ROCm/HIP available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device count:       {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_properties(i).name}")

### Custom Reasoning Format

We use a security-focused reasoning structure instead of the math `<think>` tags:
- `<start_analysis>` / `<end_analysis>` for vulnerability reasoning
- `<FIXED_CODE>` / `</FIXED_CODE>` wrapping the `java` code block

This is equivalent to DeepSeek-style chain-of-thought but for security analysis.

In [ ]:
REASONING_START = "<start_analysis>"
REASONING_END   = "<end_analysis>"
SOLUTION_START  = "<FIXED_CODE>"
SOLUTION_END    = "</FIXED_CODE>"

SYSTEM_PROMPT = (
    "You are an expert Java security engineer specialising in vulnerability remediation.\n"
    "When given a vulnerability report and vulnerable Java code:\n"
    f"1. Analyse the root cause between {REASONING_START} and {REASONING_END}.\n"
    f"2. Provide the fixed, secure Java code between {SOLUTION_START} and {SOLUTION_END}.\n"
    "The fixed code must be inside a ```java ... ``` fenced block within the solution tags.\n"
    "Preserve original logic — only change what is necessary to eliminate the vulnerability."
)

print("System prompt:")
print(SYSTEM_PROMPT)


### ⚠️ Code Confidence Analysis — Read Before Running

**Overall confidence: 6.5 / 10**

| Aspect | Status | Notes |
|---|---|---|
| Pre-SFT warmup before GRPO | ✅ Correct | Proven pattern from Unsloth; prevents format confusion |
| Custom reasoning format | ✅ Good | Security-specific tags, better than generic `<think>` |
| LoRA rank 32, alpha 64 | ✅ Appropriate | Solid for Qwen3-8B |
| Memory management | ✅ Correct | `del` + `gc.collect` + `empty_cache` between stages |
| Security reward functions | ✅ Good | Unique multi-signal design covering safe APIs, vuln removal |
| **Dataset size (143 samples)** | ⚠️ Small | GRPO typically converges with 1 000+ samples. 143 may produce noisy/slow reward growth. Consider adding `javavfc` dataset (~784 rows skipped earlier — fix parsing and include them). |
| **GRPO_MAX_STEPS = 200** | ⚠️ Low | With 143 samples, 200 steps ≈ 1.4 passes. Increase to **500–1000** for a real run. |
| `load_adapter` after `get_peft_model` | ⚠️ Verify | Loading warmup LoRA into an already-initialised PEFT model can conflict. Monitor for errors at that cell. |
| Regex-only rewards | ⚠️ Limited | Pattern matching can't verify semantic correctness. A model can pass by copying a template. |
| vLLM + Unsloth GRPO | ⚠️ Version-sensitive | `fast_inference=True` requires exact vLLM compatibility. If GRPO training hangs or crashes at generation, set `fast_inference=False` and remove `vllm_sampling_params`. |

**Quick fixes to apply now:**
1. `GRPO_MAX_STEPS = 200` → set to **500** before running for hackathon
2. `GRPO_NUM_GEN = 4` → reduce to **2** first if you hit OOM
3. If reward is still 0 after 150 steps → try `GRPO_TEMPERATURE = 1.2` to increase exploration


### Load Model + Apply Chat Template

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name             = MODEL_NAME,
    max_seq_length         = MAX_SEQ_LENGTH,
    load_in_4bit           = True,
    fast_inference         = False,   # False for SFT warmup; True needed for GRPO later
    max_lora_rank          = LORA_RANK,
    gpu_memory_utilization = 0.85,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = LORA_RANK * 2,
    lora_dropout               = 0,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)

print(model.print_trainable_parameters())

In [ ]:
# Inject custom reasoning format into the chat template
chat_template = (
    "{% if messages[0]['role'] == 'system' %}"
        "{{ messages[0]['content'] + eos_token }}"
        "{% set loop_messages = messages[1:] %}"
    "{% else %}"
        "{{ '" + SYSTEM_PROMPT.replace("'", "\\'") + "' + eos_token }}"
        "{% set loop_messages = messages %}"
    "{% endif %}"
    "{% for message in loop_messages %}"
        "{% if message['role'] == 'user' %}"
            "{{ message['content'] }}"
        "{% elif message['role'] == 'assistant' %}"
            "{{ message['content'] + eos_token }}"
        "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '" + REASONING_START + "' }}{% endif %}"
)
tokenizer.chat_template = chat_template

# Preview with a quick example
print(tokenizer.apply_chat_template([
    {"role": "user", "content": "Fix this SQL injection."},
    {"role": "assistant", "content":
        f"{REASONING_START}The code concatenates user input directly into SQL.{REASONING_END}"
        f"{SOLUTION_START}\n```java\nPreparedStatement ps = conn.prepareStatement(\"SELECT * FROM users WHERE id=?\");\n```\n{SOLUTION_END}"
    },
], tokenize=False, add_generation_prompt=False))

### Load and Reformat Dataset

We load `output/train_chat_template.jsonl` (143 Java CVE fix pairs) and wrap the assistant answers in our reasoning format structure.

In [ ]:
from datasets import Dataset

raw = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw.append(json.loads(line))

print(f"Loaded {len(raw)} records from {DATASET_PATH}")

def reformat_for_sft(obj):
    msgs = obj["messages"]
    system_msg = next((m for m in msgs if m["role"] == "system"),    None)
    user_msg   = next((m for m in msgs if m["role"] == "user"),      None)
    asst_msg   = next((m for m in msgs if m["role"] == "assistant"), None)
    if not user_msg or not asst_msg:
        return None
    # Wrap existing fix in reasoning + solution tags
    new_assistant = (
        f"{REASONING_START}\n"
        "Analysing the vulnerability pattern in the provided code and identifying "
        "the insecure API usage or logic flaw that needs remediation.\n"
        f"{REASONING_END}\n"
        f"{SOLUTION_START}\n"
        f"{asst_msg['content'].strip()}\n"
        f"{SOLUTION_END}"
    )
    return [
        {"role": "system",    "content": system_msg["content"] if system_msg else SYSTEM_PROMPT},
        {"role": "user",      "content": user_msg["content"]},
        {"role": "assistant", "content": new_assistant},
    ]

reformatted = [r for obj in raw if (r := reformat_for_sft(obj)) is not None]
print(f"Reformatted: {len(reformatted)} samples")

# Apply chat template and filter by length
MAX_SFT_LEN = MAX_SEQ_LENGTH // 2
sft_records = []
for msgs in reformatted:
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    if len(tokenizer.encode(text)) <= MAX_SFT_LEN:
        sft_records.append({"text": text, "messages": msgs})

sft_dataset = Dataset.from_list(sft_records)
print(f"SFT dataset after length filter (<= {MAX_SFT_LEN} tokens): {len(sft_dataset)} samples")
sft_dataset

In [ ]:
# Preview one sample
print(sft_dataset[0]["text"][:1500])

### Step 1 — Pre-SFT Warmup

Fine-tune for a few epochs on our 143-sample dataset so the model learns to:
- Use `<start_analysis>` / `<end_analysis>` for vulnerability reasoning
- Output `<FIXED_CODE>```java...```</FIXED_CODE>` for the fix

In [ ]:

from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

throughput_cb = ThroughputCallback(log_every=10)
GPUMonitor.print_stats("before_sft")

sft_trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = sft_dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps                = 5,
        num_train_epochs            = SFT_EPOCHS,
        learning_rate               = SFT_LR,
        logging_steps               = 5,
        # adamw_torch_fused: pure-PyTorch fused AdamW — stable on ROCm
        # (adamw_8bit requires bitsandbytes which can be unstable on some ROCm versions)
        optim                       = "adamw_torch_fused",
        weight_decay                = 0.001,
        lr_scheduler_type           = "linear",
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        seed                        = 42,
        output_dir                  = OUTPUT_DIR + "_sft_warmup",
        report_to                   = "none",
    ),
    callbacks = [throughput_cb],
)

sft_trainer.train()

print(f"\nSFT warmup throughput summary ({len(throughput_cb.history)} log points):")
if throughput_cb.history:
    avg_sps  = sum(r["steps_per_sec"]   for r in throughput_cb.history) / len(throughput_cb.history)
    avg_smps = sum(r["samples_per_sec"] for r in throughput_cb.history) / len(throughput_cb.history)
    peak     = max(r["peak_vram_gb"]    for r in throughput_cb.history)
    print(f"  Avg throughput: {avg_sps:.2f} steps/s | {avg_smps:.2f} samples/s")
    print(f"  Peak VRAM:      {peak:.1f} GB")


In [ ]:
# Sanity check — does the model follow the reasoning format?
from transformers import TextStreamer

sample = sft_dataset[0]["messages"][:2]  # system + user only
text = tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

print("--- Format check (first 400 new tokens) ---")
_ = model.generate(
    **inputs,
    max_new_tokens = 400,
    temperature    = 0.3,
    do_sample      = True,
    streamer       = TextStreamer(tokenizer, skip_prompt=True),
    pad_token_id   = tokenizer.eos_token_id,
)

In [ ]:
# Save the SFT warmup LoRA, then free memory before GRPO
warmup_path = OUTPUT_DIR + "_sft_warmup/lora_adapter"
model.save_pretrained(warmup_path)
tokenizer.save_pretrained(warmup_path)
print(f"SFT warmup LoRA saved → {warmup_path}")

del sft_trainer, sft_dataset
torch.cuda.empty_cache()
gc.collect()
print("Memory freed.")

### Step 2a — Prepare GRPO Dataset

GRPO format: `{"prompt": [system+user msgs], "gt_fix": "..."}`.  
The model generates the assistant turn; reward functions judge the output.

In [ ]:
from datasets import Dataset as HFDataset

grpo_records = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        msgs = obj["messages"]
        system_msg = next((m for m in msgs if m["role"] == "system"), None)
        user_msg   = next((m for m in msgs if m["role"] == "user"),   None)
        asst_msg   = next((m for m in msgs if m["role"] == "assistant"), None)
        if not user_msg or not asst_msg:
            continue
        grpo_records.append({
            "prompt": [
                {"role": "system", "content": system_msg["content"] if system_msg else SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg["content"]},
            ],
            "gt_fix": asst_msg["content"],
        })

grpo_dataset = HFDataset.from_list(grpo_records)

# Filter by prompt length
max_prompt_len = MAX_SEQ_LENGTH // 2
prompt_lens = [
    len(tokenizer.apply_chat_template(r["prompt"], add_generation_prompt=True, tokenize=True))
    for r in grpo_records
]
grpo_dataset = grpo_dataset.select([i for i, l in enumerate(prompt_lens) if l <= max_prompt_len])
print(f"GRPO dataset: {len(grpo_dataset)} samples (max prompt len = {max_prompt_len})")

### Step 2b — Define Security-Aware Reward Functions

| Reward function | Signal | Range |
|---|---|---|
| `reward_format_exact` | Full `<start_analysis>...<FIXED_CODE>` structure present | 0 / +3 |
| `reward_format_approximate` | Partial tags present | -1.5 to +1.5 |
| `reward_has_java_block` | ` ```java ``` ` block in solution | -1 / +2 |
| `reward_no_vuln_patterns` | -1 per vuln API still in fixed code | ≤ 0 |
| `reward_uses_safe_apis` | +0.5 per safe API (capped +2) | 0 to +2 |
| `reward_code_actually_changed` | Fixed code differs from vulnerable code | -1.5 / +1.5 |

In [ ]:
VULN_PATTERNS = [
    r'Runtime\.getRuntime\(\)\.exec\(',
    r'(?<!\w)Statement\b.*\.execute(?:Query|Update)?\s*\(\s*["\']',
    r'new\s+File\s*\(\s*(?:request\.|userInput)',
    r'ObjectInputStream\s*\(',
    r'MessageDigest\.getInstance\s*\(\s*["\'](?:MD5|SHA-1)["\']',
    r'new\s+Random\s*\(\s*\)',
]

SAFE_PATTERNS = [
    r'PreparedStatement',
    r'\.setString\s*\(\d+',
    r'\.setInt\s*\(\d+',
    r'ProcessBuilder',
    r'\.toPath\(\)\.normalize\(',
    r'Paths\.get\(',
    r'SecureRandom',
    r'MessageDigest\.getInstance\s*\(\s*["\']SHA-(?:256|512)["\']',
    r'Base64\.getEncoder\(',
    r'HtmlUtils\.htmlEscape\(',
    r'StringEscapeUtils',
]

SOLUTION_END_RE = re.escape(SOLUTION_END) + r"[\s]{0,}" + r"(?:<\|im_end\|>|</s>)?"
MATCH_FORMAT = re.compile(
    rf"{re.escape(REASONING_END)}.*?"
    rf"{re.escape(SOLUTION_START)}(.+?){SOLUTION_END_RE}"
    r"[\s]{0,}$",
    flags=re.MULTILINE | re.DOTALL,
)
MATCH_JAVA = re.compile(r'```java\s*([\s\S]+?)```', re.DOTALL)

def _sol(text): m = MATCH_FORMAT.search(text); return m.group(1).strip() if m else ""
def _java(text): m = MATCH_JAVA.search(text); return m.group(1).strip() if m else ""

# Verify regex on a crafted example
test = (
    f"{REASONING_START}The code has SQL injection.{REASONING_END}"
    f"{SOLUTION_START}\n```java\nPreparedStatement ps = conn.prepareStatement(\"SELECT * FROM users WHERE id=?\");\n```\n{SOLUTION_END}"
)
print("Format match:", bool(MATCH_FORMAT.search(test)))
print("Java extract:", _java(_sol(test))[:80])

In [ ]:
def reward_format_exact(completions, **kwargs):
    return [3.0 if MATCH_FORMAT.search(c[0]["content"]) else 0.0 for c in completions]

def reward_format_approximate(completions, **kwargs):
    scores = []
    for c in completions:
        r = c[0]["content"]
        score  = 0.5 if r.count(REASONING_END)  == 1 else -0.5
        score += 0.5 if r.count(SOLUTION_START) == 1 else -0.5
        score += 0.5 if r.count(SOLUTION_END)   == 1 else -0.5
        scores.append(score)
    return scores

def reward_has_java_block(completions, **kwargs):
    scores = []
    for c in completions:
        solution = _sol(c[0]["content"]) or c[0]["content"]
        scores.append(2.0 if "```java" in solution else -1.0)
    return scores

def reward_no_vuln_patterns(completions, **kwargs):
    scores = []
    for c in completions:
        code = _java(_sol(c[0]["content"])) or c[0]["content"]
        hits = sum(1 for p in VULN_PATTERNS if re.search(p, code))
        scores.append(-float(hits))
    return scores

def reward_uses_safe_apis(completions, **kwargs):
    scores = []
    for c in completions:
        code = _java(_sol(c[0]["content"])) or c[0]["content"]
        hits = sum(0.5 for p in SAFE_PATTERNS if re.search(p, code))
        scores.append(min(hits, 2.0))
    return scores

def reward_code_actually_changed(completions, prompts, **kwargs):
    scores = []
    for c, prompt_msgs in zip(completions, prompts):
        fixed = _java(_sol(c[0]["content"])) or _java(c[0]["content"])
        if not fixed:
            scores.append(-0.5)
            continue
        user_content = next((m["content"] for m in prompt_msgs if m["role"] == "user"), "")
        vuln = _java(user_content)
        scores.append(-1.5 if (vuln and fixed.strip() == vuln.strip()) else 1.5)
    return scores

REWARD_FUNCTIONS = [
    reward_format_exact,
    reward_format_approximate,
    reward_has_java_block,
    reward_no_vuln_patterns,
    reward_uses_safe_apis,
    reward_code_actually_changed,
]
print(f"{len(REWARD_FUNCTIONS)} reward functions defined.")

### Step 2c — GRPO Training

Watch the `reward` column in the training log — it should climb after ~50-100 steps.

**Max score per step = 3 + 1.5 + 2 + 0 + 2 + 1.5 = 10.0** (all rewards hit)

In [ ]:
# Reload model with fast_inference=True (required for vLLM inside GRPO)
del model
torch.cuda.empty_cache()
gc.collect()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name             = MODEL_NAME,
    max_seq_length         = MAX_SEQ_LENGTH,
    load_in_4bit           = True,
    fast_inference         = True,   # vLLM for GRPO generation
    max_lora_rank          = LORA_RANK,
    gpu_memory_utilization = 0.85,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = LORA_RANK * 2,
    lora_dropout               = 0,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)

# Re-apply chat template (tokenizer was reloaded)
tokenizer.chat_template = chat_template

# Load SFT warmup weights
warmup_path = OUTPUT_DIR + "_sft_warmup/lora_adapter"
if Path(warmup_path).exists():
    model.load_adapter(warmup_path, adapter_name="default")
    print(f"Loaded SFT warmup LoRA from {warmup_path}")
else:
    print("[WARN] SFT warmup LoRA not found — starting GRPO from base model.")

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from unsloth import is_bfloat16_supported
from vllm import SamplingParams

vllm_sampling_params = SamplingParams(
    min_p                     = 0.05,
    top_p                     = 0.95,
    top_k                     = -1,
    seed                      = 42,
    stop                      = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

grpo_config = GRPOConfig(
    vllm_sampling_params        = vllm_sampling_params,
    temperature                 = GRPO_TEMPERATURE,
    learning_rate               = GRPO_LR,
    adam_beta1                  = 0.9,
    adam_beta2                  = 0.99,
    weight_decay                = 0.1,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = "cosine",
    # adamw_torch_fused: pure-PyTorch fused AdamW — stable on ROCm
    # (adamw_8bit requires bitsandbytes which can be unstable on some ROCm versions)
    optim                       = "adamw_torch_fused",
    logging_steps               = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations             = GRPO_NUM_GEN,
    max_prompt_length           = MAX_SEQ_LENGTH // 2,
    max_completion_length       = GRPO_MAX_NEW_TOKENS,
    max_steps                   = GRPO_MAX_STEPS,
    save_steps                  = 50,
    save_total_limit            = 3,
    output_dir                  = OUTPUT_DIR,
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    report_to                   = "none",
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = REWARD_FUNCTIONS,
    args             = grpo_config,
    train_dataset    = grpo_dataset,
)

print(f"Starting GRPO for {GRPO_MAX_STEPS} steps with {len(grpo_dataset)} samples...")
print("Tip: reward should start rising after ~50-100 steps.")
trainer.train()

In [ ]:
# Save GRPO LoRA adapter
lora_path = OUTPUT_DIR + "/lora_adapter"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"GRPO LoRA adapter saved → {lora_path}")

### Inference Test

Test with classic SQL injection and path traversal examples.

In [ ]:
from unsloth import FastLanguageModel
from transformers import TextStreamer

FastLanguageModel.for_inference(model)  # 2x faster inference

def ask_model(vuln_report: str, vuln_code: str, max_tokens: int = 600):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"## Security Vulnerability Report\n\n{vuln_report}\n\n"
                f"## Vulnerable Java Code\n\n```java\n{vuln_code}\n```\n\n"
                "Please provide the fixed, secure version of this code."
            ),
        },
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    print("\n" + "─" * 60)
    print("MODEL OUTPUT:")
    print("─" * 60)
    _ = model.generate(
        **inputs,
        max_new_tokens = max_tokens,
        temperature    = 0.3,
        do_sample      = True,
        streamer       = TextStreamer(tokenizer, skip_prompt=True),
        pad_token_id   = tokenizer.eos_token_id,
    )
    print("─" * 60)

print("Inference helper ready.")

In [ ]:
# Test 1: SQL Injection
ask_model(
    vuln_report = "**CVE ID:** CVE-2021-EXAMPLE\n**Weakness:** CWE-089 — SQL Injection\n**Vulnerability Type:** SQL Injection",
    vuln_code   = (
        'public List<User> findByUsername(String username) {\n'
        '    String sql = "SELECT * FROM users WHERE username = \'" + username + "\'";\n'
        '    return jdbcTemplate.query(sql, new UserRowMapper());\n'
        '}'
    ),
)

In [ ]:
# Test 2: Path Traversal
ask_model(
    vuln_report = "**Weakness:** CWE-022 — Path Traversal\n**Vulnerability Type:** Path Traversal",
    vuln_code   = (
        'public void downloadFile(HttpServletRequest request, HttpServletResponse response) throws IOException {\n'
        '    String filename = request.getParameter("file");\n'
        '    File file = new File("/uploads/" + filename);\n'
        '    Files.copy(file.toPath(), response.getOutputStream());\n'
        '}'
    ),
)

### Save Merged Model for vLLM Deployment

Merges the LoRA adapter back into the base weights and saves as float16 — ready for `vllm serve`.

In [ ]:

### Export & Quantize

Choose which formats to export by editing `EXPORT_FORMATS` in the next cell.

| Format | File size (8B model) | Use case |
|---|---|---|
| `q4_k_s` | ~3.5 GB | Smallest / fastest, slight quality loss |
| `q4_k_m` | ~4.1 GB | **Recommended** — best balance |
| `q4_k_l` | ~4.9 GB | Best q4 quality |
| `q8_0` | ~7.5 GB | Near-lossless, ~2× larger than q4 |
| `f16` (GGUF) | ~14 GB | Lossless GGUF, for llama.cpp |
| `merged_16bit` | ~14 GB | Float16 merged model — for **vLLM serving** |


In [ ]:

import time as _t

# ─── Edit this list to choose your export formats ────────────────────────────
EXPORT_FORMATS = [
    "q4_k_m",        # recommended GGUF — best size/quality balance
    # "q4_k_s",      # smallest GGUF (~3.5 GB for 8B)
    # "q4_k_l",      # best q4 quality GGUF (~4.9 GB for 8B)
    # "q8_0",        # near-lossless GGUF (~7.5 GB for 8B)
    # "f16",         # lossless GGUF float16 (~14 GB)
    # "merged_16bit",# merged float16 model — for vLLM serving
]
# ─────────────────────────────────────────────────────────────────────────────

gguf_formats   = [m for m in EXPORT_FORMATS if not m.startswith("merged")]
merged_formats = [m for m in EXPORT_FORMATS if m.startswith("merged")]

GPUMonitor.print_stats("before_export")

# ── GGUF export (llama.cpp / Ollama compatible) ───────────────────────────────
if gguf_formats:
    gguf_out = OUTPUT_DIR + "_gguf"
    print(f"\nExporting GGUF [{', '.join(gguf_formats)}] → {gguf_out}")
    t0 = _t.time()
    model.save_pretrained_gguf(gguf_out, tokenizer, quantization_method=gguf_formats)
    elapsed = _t.time() - t0
    print(f"  ✓ GGUF export done in {elapsed:.1f}s")
    print(f"\nFiles created in {gguf_out}/:")
    from pathlib import Path as _P
    for f in sorted(_P(gguf_out).glob("*.gguf")):
        size_gb = f.stat().st_size / 1024**3
        print(f"  {f.name:<55} {size_gb:.2f} GB")
    print(f"\nTo use with llama.cpp:")
    print(f"  ./llama-cli -m {gguf_out}/<file>.gguf --chat-template chatml")
    print(f"\nTo use with Ollama:")
    print(f"  ollama create java-vuln-fix --from {gguf_out}/<file>.gguf")

# ── Merged float16 export (vLLM serving) ────────────────────────────────────
for method in merged_formats:
    merged_out = OUTPUT_DIR + f"_{method}"
    print(f"\nExporting {method} → {merged_out}")
    t0 = _t.time()
    model.save_pretrained_merged(merged_out, tokenizer, save_method=method)
    elapsed = _t.time() - t0
    print(f"  ✓ {method} export done in {elapsed:.1f}s")
    print(f"\nTo serve with vLLM:")
    print(f"  vllm serve {merged_out} --max-model-len 4096 --dtype bfloat16")
    print(f"  python vllm-chat.py --url http://localhost:8000")
    print(f"\nTo benchmark:")
    print(f"  python modelcompare-pre-post.py --model {merged_out} --mode local --output results/finetuned.json")

GPUMonitor.print_stats("after_export")
print("\n✓ Export complete.")
